# Perspective Correction + GLB Generation

Takes the Nova Canvas cutouts, corrects perspective using Claude to detect corners, then generates a double-sided plane GLB for AR viewing.

In [ ]:
!pip install -q boto3 pillow matplotlib opencv-python-headless numpy trimesh pygltflib

In [ ]:
import os
import json
import base64
import numpy as np
import cv2
import boto3
import trimesh
from PIL import Image
import matplotlib.pyplot as plt

bedrock = boto3.client("bedrock-runtime", region_name="us-west-2")
# Change model ID to whatever worked for you in the previous notebook
MODEL_ID = "us.anthropic.claude-3-5-sonnet-20241022-v2:0"

In [ ]:
# Point to your cutout images
FRONT_IMAGE = "output/WhatsApp Image 2026-05-13 at 9.07.58 AM_cutout.png"
BACK_IMAGE = "output/WhatsApp Image 2026-05-13 at 9.07.58 AM (2)_cutout.png"

# Door dimensions (mm) - adjust to your actual door
DOOR_WIDTH_MM = 900
DOOR_HEIGHT_MM = 2100

In [ ]:
def get_door_corners(image_path):
    """Ask Claude to identify the 4 corners of the door panel."""
    with open(image_path, "rb") as f:
        img_b64 = base64.b64encode(f.read()).decode()

    img = Image.open(image_path)
    w, h = img.size

    response = bedrock.invoke_model(
        modelId=MODEL_ID,
        body=json.dumps({
            "anthropic_version": "bedrock-2023-05-31",
            "max_tokens": 300,
            "messages": [{"role": "user", "content": [
                {"type": "image", "source": {"type": "base64", "media_type": "image/png", "data": img_b64}},
                {"type": "text", "text": f"This is a cutout of a door on a white/transparent background. Image is {w}x{h} pixels. Identify the 4 corners of the door PANEL (the flat rectangular surface, not the frame). Return ONLY JSON: {{\"corners\": [[x1,y1],[x2,y2],[x3,y3],[x4,y4]]}} in order: top-left, top-right, bottom-right, bottom-left. No explanation."}
            ]}]
        }),
        accept="application/json",
        contentType="application/json"
    )

    body = json.loads(response["body"].read())
    text = body["content"][0]["text"]
    result = json.loads(text)
    corners = np.array(result["corners"], dtype=np.float32)
    print(f"  Corners: {corners.tolist()}")
    return corners

In [ ]:
def perspective_correct(image_path, corners, output_width=900, output_height=2100):
    """Warp door to a rectangle using homography."""
    img = cv2.imread(image_path, cv2.IMREAD_UNCHANGED)  # keep alpha

    # Destination corners (perfect rectangle)
    dst = np.array([
        [0, 0],
        [output_width - 1, 0],
        [output_width - 1, output_height - 1],
        [0, output_height - 1]
    ], dtype=np.float32)

    # Compute homography and warp
    M = cv2.getPerspectiveTransform(corners, dst)
    warped = cv2.warpPerspective(img, M, (output_width, output_height))

    return warped

In [ ]:
# Process front
print("Front door:")
front_corners = get_door_corners(FRONT_IMAGE)
front_warped = perspective_correct(FRONT_IMAGE, front_corners)

plt.figure(figsize=(4, 10))
plt.imshow(cv2.cvtColor(front_warped, cv2.COLOR_BGRA2RGBA))
plt.title("Front (corrected)")
plt.axis("off")
plt.show()

cv2.imwrite("output/front_corrected.png", front_warped)
print("Saved: output/front_corrected.png")

In [ ]:
# Process back
print("Back door:")
back_corners = get_door_corners(BACK_IMAGE)
back_warped = perspective_correct(BACK_IMAGE, back_corners)

plt.figure(figsize=(4, 10))
plt.imshow(cv2.cvtColor(back_warped, cv2.COLOR_BGRA2RGBA))
plt.title("Back (corrected)")
plt.axis("off")
plt.show()

cv2.imwrite("output/back_corrected.png", back_warped)
print("Saved: output/back_corrected.png")

In [ ]:
def build_door_glb(front_path, back_path, output_path, width_m=0.9, height_m=2.1):
    """Build a double-sided plane GLB with front/back textures."""
    # Load textures
    front_img = Image.open(front_path).convert("RGBA")
    back_img = Image.open(back_path).convert("RGBA")
    # Flip back image horizontally (mirror) so it looks correct from behind
    back_img = back_img.transpose(Image.FLIP_LEFT_RIGHT)

    hw = width_m / 2
    hh = height_m / 2

    # Front face (facing +Z)
    front_vertices = np.array([
        [-hw, -hh, 0],
        [ hw, -hh, 0],
        [ hw,  hh, 0],
        [-hw,  hh, 0],
    ], dtype=np.float32)
    front_faces = np.array([[0, 1, 2], [0, 2, 3]])
    front_uv = np.array([[0, 1], [1, 1], [1, 0], [0, 0]], dtype=np.float32)

    # Back face (facing -Z) — reversed winding
    back_vertices = np.array([
        [ hw, -hh, 0],
        [-hw, -hh, 0],
        [-hw,  hh, 0],
        [ hw,  hh, 0],
    ], dtype=np.float32)
    back_faces = np.array([[0, 1, 2], [0, 2, 3]])
    back_uv = np.array([[0, 1], [1, 1], [1, 0], [0, 0]], dtype=np.float32)

    # Create front mesh
    front_material = trimesh.visual.material.PBRMaterial(
        baseColorTexture=front_img,
        alphaMode='OPAQUE'
    )
    front_visual = trimesh.visual.TextureVisuals(
        uv=front_uv, material=front_material
    )
    front_mesh = trimesh.Trimesh(
        vertices=front_vertices, faces=front_faces, visual=front_visual
    )

    # Create back mesh
    back_material = trimesh.visual.material.PBRMaterial(
        baseColorTexture=back_img,
        alphaMode='OPAQUE'
    )
    back_visual = trimesh.visual.TextureVisuals(
        uv=back_uv, material=back_material
    )
    back_mesh = trimesh.Trimesh(
        vertices=back_vertices, faces=back_faces, visual=back_visual
    )

    # Combine into scene
    scene = trimesh.Scene()
    scene.add_geometry(front_mesh, node_name="front")
    scene.add_geometry(back_mesh, node_name="back")

    # Export as GLB
    scene.export(output_path, file_type="glb")
    print(f"GLB saved: {output_path}")
    print(f"  Size: {os.path.getsize(output_path) / 1024:.1f} KB")

In [ ]:
# Build the GLB
build_door_glb(
    "output/front_corrected.png",
    "output/back_corrected.png",
    "output/door.glb",
    width_m=DOOR_WIDTH_MM / 1000,
    height_m=DOOR_HEIGHT_MM / 1000
)

## Done!

Download `output/door.glb` and test it:
- Drag into https://gltf-viewer.donmccurdy.com/ to preview
- Or use it in the webapp with `<model-viewer>`

The GLB is a double-sided flat plane — front texture on one side, back on the other. When you orbit around it in AR/3D viewer, you'll see both faces.